1.Load data

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
# set visual style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
#Load the data from web(sometimes, the web may be down. go for downloading option)
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
online_retail = fetch_ucirepo(id=352)
df = online_retail.data.original
df.head()

In [ ]:
#Load by downloading the file
df = pd.read_excel('Users/jinhong.wu/Online Retail.xlsx')

2. data overview

In [ ]:
#Explore the data
print(f"\nDataset Shape: {df.shape}")
print("="*60)
print("Column names and data types")
print(df.dtypes)
print("="*60)
print("First 5 rows")
print(df.head(5))
print("="*60)
print("Basic statistics")
print(df.describe())
print("="*60)
print("Missing values")
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_values,'Percentage': missing_percent})
print(missing_df[missing_df['Missing_Count'] > 0])
print("="*60)
print("Unique value per column")
for col in df.columns:
    print(f"{col}: {df[col].nunique():,} unique values")
print("="*60)
duplicate_count = df.duplicated().sum()
print(f"Total exact duplicate rows found: {duplicate_count}")

3. data cleaning, data preprocessing, feature engineering, EDA

In [ ]:
#data cleaning and preprocessing
df_clean = df.copy()
# 1.convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
#2.create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
#3.identify and remove cancellations
df_clean['IsCancellation'] = df_clean['InvoiceNo'].astype(str).str.startswith('C')
df_clean=df_clean[df_clean['IsCancellation'] == False]
# 4.handle missing values in customerid
df_clean = df_clean[df_clean['CustomerID'].notna()]
#5. handle data quality: quantity, unitprice
df_clean = df_clean[df_clean['UnitPrice']>0]
df_clean = df_clean[df_clean['Quantity']>0]
#feature engineering
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek  # 0=Monday, 6=Sunday
df_clean['Date'] = df_clean['InvoiceDate'].dt.date
#remove duplicate
df_clean = df_clean.drop_duplicates()

print("CLEANED DATASET SUMMARY")
print("="*60)
retention_rate = (len(df_clean) / len(df)) * 100
print(f"Data retention: {retention_rate:.2f}%")
print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Shape: {df_clean.shape}")
print(f"Date Range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
print(f"Unique Customers: {df_clean['CustomerID'].nunique():,}")
print(f"Unique Products: {df_clean['StockCode'].nunique():,}")
print(f"Unique Invoices: {df_clean['InvoiceNo'].nunique():,}")
print(f"Countries: {df_clean['Country'].nunique()}")
print(f"Total Revenue: £{df_clean['TotalPrice'].sum():,.2f}")

In [ ]:
#EDA
# set up visual style
plt.style.use('seaborn-v0_8-darkgrid')
colors = ['#45B7D1', '#FFA07A','#98D8C8','#FF6B6B', '#4ECDC4']

#1.Distribution of Quantity, Unit Price, Total amount
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
fig.suptitle('Quantity, Unit Price and Total Amount Distribution Analysis', fontsize=16, fontweight='bold')
#quantity
axes[0, 0].hist(df_clean['Quantity'], bins=50, color=colors[0], edgecolor='black', alpha=0.8)
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_xlabel('Quantity')
axes[0, 0].set_title('Quantity Distribution')
axes[0, 0].axvline(df_clean['Quantity'].median(), color='red', linestyle='--', label=f'Median: {df_clean["Quantity"].median():.0f}')
axes[0, 0].legend()
#quantity(log scale for better view)
axes[0, 1].hist(df_clean['Quantity'], bins=50, color=colors[0], edgecolor='black', alpha=0.8)
axes[0, 1].set_yscale('log')
axes[0, 1].set_ylabel('Frequency (log)')
axes[0, 1].set_xlabel('Quantity')
axes[0, 1].set_title('Quantity Distribution (Log Scale)')
# unit price
axes[1, 0].hist(df_clean['UnitPrice'], bins=50, color=colors[1], edgecolor='black', alpha=0.8)
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_xlabel('Unit Price (£)')
axes[1, 0].set_title('Unit Price Distribution')
axes[1, 0].axvline(df_clean['UnitPrice'].median(), color='red', linestyle='--', label=f'Median: £{df_clean["UnitPrice"].median():.2f}')
axes[1, 0].legend()
#unit price (zoom in:<£50)
price_filtered = df_clean[df_clean['UnitPrice'] < 50]
axes[1, 1].hist(price_filtered['UnitPrice'], bins=50, color=colors[1], edgecolor='black', alpha=0.8)
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_xlabel('Unit Price (£)')
axes[1, 1].set_title('Unit Price Distribution (<£50)')
#total price
axes[2, 0].hist(df_clean['TotalPrice'], bins=50, color=colors[2], edgecolor='black', alpha=0.8)
axes[2, 0].set_ylabel('Frequency')
axes[2, 0].set_xlabel('Total Amount (£)')
axes[2, 0].set_title('Total Transaction Amount Distribution')
axes[2, 0].axvline(df_clean['TotalPrice'].median(), color='red', linestyle='--', label=f'Median: £{df_clean["TotalPrice"].median():.2f}')
axes[2, 0].legend()
#total price(log scale)
axes[2, 1].hist(df_clean['TotalPrice'], bins=50, color=colors[2], edgecolor='black', alpha=0.8)
axes[2, 1].set_yscale('log')
axes[2, 1].set_ylabel('Frequency (log)')
axes[2, 1].set_xlabel('Total Amount (£)')
axes[2, 1].set_title('Total Amount Distribution (Log Scale)')

plt.tight_layout()
plt.show()

#statistics
print("\nQuantity, Unit Price and Total Amount Distribution Statistics:")
print(f"Quantity- Mean: {df_clean['Quantity'].mean():.2f}, Median: {df_clean['Quantity'].median():.2f}, Std: {df_clean['Quantity'].std():.2f}")
print(f"UnitPrice- Mean: £{df_clean['UnitPrice'].mean():.2f}, Median: £{df_clean['UnitPrice'].median():.2f}, Std: £{df_clean['UnitPrice'].std():.2f}")
print(f"TotalPrice- Mean: £{df_clean['TotalPrice'].mean():.2f}, Median: £{df_clean['TotalPrice'].median():.2f}, Std: £{df_clean['TotalPrice'].std():.2f}")

In [ ]:
#EDA
#2.Transaction count by Country, Revenue by Country
country_count = df_clean.groupby('Country').agg({'InvoiceNo': 'count'}).reset_index()
country_count.columns = ['Country', 'Transaction_count']
country_count = country_count.sort_values('Transaction_count', ascending=False)
country_amt = df_clean.groupby('Country').agg({'TotalPrice': 'sum'}).reset_index()
country_amt.columns = ['Country', 'Total_revenue']
country_amt = country_amt.sort_values('Total_revenue', ascending=False)
country_cust = df_clean.groupby('Country').agg({'CustomerID': 'nunique'}).reset_index()
country_cust.columns = ['Country', 'Unique_customers']
country_cust = country_cust.sort_values('Unique_customers', ascending=False)

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
fig.suptitle('Geographic analysis', fontsize=16, fontweight='bold')
#Top 10 countries by transaction count
top10_trans = country_count.head(10)
axes[0].barh(top10_trans['Country'], top10_trans['Transaction_count'], color=colors[3])
axes[0].invert_yaxis()
axes[0].set_xlabel('Number of Transactions')
axes[0].set_title('Top 10 Countries by Transaction Count')
#Top 10 countries by revenue
top10_rev = country_amt.head(10)
axes[1].barh(top10_rev['Country'], top10_rev['Total_revenue'], color=colors[4])
axes[1].invert_yaxis()
axes[1].set_xlabel('Total Revenue (£)')
axes[1].set_title('Top 15 Countries by Revenue')
#Top 10 countries by transaction count
top10_cust = country_cust.head(10)
axes[2].barh(top10_cust['Country'], top10_cust['Unique_customers'], color=colors[0])
axes[2].invert_yaxis()
axes[2].set_xlabel('Number of Customers')
axes[2].set_title('Top 10 Countries by Customer Count')
plt.tight_layout()
plt.show()

print("\nTop 10 Countries by Transaction Count:")
print(country_count.head(10)[['Country', 'Transaction_count']].to_string(index=False))
print("\nTop 10 Countries by Revenue (£):")
print(country_amt.head(10)[['Country', 'Total_revenue']].to_string(index=False))
print("\nTop 10 Countries by Customer Number:")
print(country_cust.head(10)[['Country', 'Unique_customers']].to_string(index=False))

In [ ]:
#EDA
#Transaction volume temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Temporal Transaction Volume Patterns', fontsize=16, fontweight='bold')
#daily-transaction count
daily_trans = df_clean.groupby('Date').agg({'InvoiceNo':'count','TotalPrice':'sum'}).reset_index()
daily_trans.columns = ['Date', 'Transactions', 'Revenue']
axes[0, 0].plot(daily_trans['Date'], daily_trans['Transactions'], color=colors[1], linewidth=1)
axes[0, 0].set_title('Daily Transaction Volume')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Number of Transactions')
axes[0, 0].tick_params(axis='x', rotation=45)
#daily-revenue
axes[0, 1].plot(daily_trans['Date'], daily_trans['Revenue'], color=colors[2], linewidth=1)
axes[0, 1].set_title('Daily Revenue')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Revenue (£)')
axes[0, 1].tick_params(axis='x', rotation=45)
#day of week pattern-transaction count
dow_trans = df_clean.groupby('DayOfWeek').agg({'InvoiceNo':'count', 'TotalPrice':'sum'}).reset_index()
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_trans['DayName'] = dow_trans['DayOfWeek'].map(dict(enumerate(dow_names)))
axes[1, 0].bar(dow_trans['DayName'], dow_trans['InvoiceNo'], color=colors[3])
axes[1, 0].set_title('Transaction Volume by Day of Week')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Number of Transactions')
axes[1, 0].tick_params(axis='x', rotation=45)
#day of week pattern-revenue
axes[1, 1].bar(dow_trans['DayName'], dow_trans['TotalPrice'], color=colors[4])
axes[1, 1].set_title('Revenue by Day of Week')
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Revenue (£)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
#EDA
#Top products analysis by revenue and purchase frequency
product_stats = df_clean.groupby(['StockCode', 'Description']).agg({'TotalPrice':'sum','Quantity':'sum','InvoiceNo': 'count'}).reset_index()
product_stats.columns = ['StockCode', 'Description', 'Total Revenue (£)', 'Total Quantity', 'Frequency']
product_stats = product_stats.sort_values('Total Revenue (£)', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('Top Products Analysis', fontsize=16, fontweight='bold')
#Top 10 by revenue
top10_prod_rev = product_stats.head(10)
axes[0].barh(range(len(top10_prod_rev)), top10_prod_rev['Total Revenue (£)'], color=colors[4])
axes[0].invert_yaxis()
axes[0].set_yticks(range(len(top10_prod_rev)))
axes[0].set_yticklabels([desc[:40] + '...' if len(desc) > 40 else desc for desc in top10_prod_rev['Description']], fontsize=9)
axes[0].set_xlabel('Total Revenue (£)')
axes[0].set_title('Top 10 Products by Revenue')
#Top 10 by purchase frequency
top10_prod_freq = product_stats.sort_values('Frequency', ascending=False).head(10)
axes[1].barh(range(len(top10_prod_freq)), top10_prod_freq['Frequency'], color=colors[0])
axes[1].invert_yaxis()
axes[1].set_yticks(range(len(top10_prod_freq)))
axes[1].set_yticklabels([desc[:40] + '...' if len(desc) > 40 else desc for desc in top10_prod_freq['Description']], fontsize=9)
axes[1].set_xlabel('Purchase Frequency')
axes[1].set_title('Top 10 Products by Purchase Frequency')

plt.tight_layout()
plt.show()

print("\nTop 10 Products by Revenue:")
print(product_stats.head(10)[['Description', 'Total Revenue (£)','Frequency']].to_string(index=False))

In [ ]:
#EDA+feature engineering/data preprocessing
#RFM
#calculate RFM
#Define the reference date (last date in dataset + 1 day)
reference_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"RFM Reference Date: {reference_date}")

#calculate RFM for each customer
rfm = df_clean.groupby('CustomerID').agg({'InvoiceDate':lambda x:(reference_date - x.max()).days,# Recency
    'InvoiceNo': 'nunique',  # Frequency (unique invoices)
    'TotalPrice': 'sum'  # Monetary
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

#RFM Stats
print("="*60)
print("RFM Statistics:")
print("Recency (days since last purchase):")
print(f"   Mean: {rfm['Recency'].mean():.2f} days")
print(f"   Median: {rfm['Recency'].median():.2f} days")
print(f"   Min: {rfm['Recency'].min()} days, Max: {rfm['Recency'].max()} days")

print("Frequency (number of purchases):")
print(f"   Mean: {rfm['Frequency'].mean():.2f} purchases")
print(f"   Median: {rfm['Frequency'].median():.2f} purchases")
print(f"   Min: {rfm['Frequency'].min()}, Max: {rfm['Frequency'].max()}")

print("Monetary (total spend):")
print(f"   Mean: £{rfm['Monetary'].mean():,.2f}")
print(f"   Median: £{rfm['Monetary'].median():,.2f}")
print(f"   Min: £{rfm['Monetary'].min():.2f}, Max: £{rfm['Monetary'].max():,.2f}")
print("="*60)

#RFM visuals
fig, axes = plt.subplots(3, 2, figsize=(12, 18))
fig.suptitle('RFM Distribution Analysis', fontsize=16, fontweight='bold')
#RECENCY
# Recency histogram
axes[0, 0].hist(rfm['Recency'], bins=50, color=colors[1], edgecolor='black', alpha=0.8)
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_xlabel('Days Since Last Purchase')
axes[0, 0].set_title('Recency Distribution')
axes[0, 0].axvline(rfm['Recency'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {rfm["Recency"].median():.0f}')
axes[0, 0].legend()
# Recency boxplot
axes[0, 1].boxplot(rfm['Recency'], vert=True, patch_artist=True, boxprops=dict(facecolor=colors[1], alpha=0.8))
axes[0, 1].set_ylabel('Days Since Last Purchase')
axes[0, 1].set_title('Recency Boxplot')
#FREQUENCY
# Frequency histogram
axes[1, 0].hist(rfm['Frequency'], bins=50, color=colors[2], edgecolor='black', alpha=0.8)
axes[1, 0].set_ylabel('Number of Customers')
axes[1, 0].set_xlabel('Number of Purchases')
axes[1, 0].set_title('Frequency Distribution')
axes[1, 0].axvline(rfm['Frequency'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {rfm["Frequency"].median():.0f}')
axes[1, 0].legend()
# Frequency histogram (log scale for better view)
axes[1, 1].hist(rfm['Frequency'], bins=50, color=colors[2], edgecolor='black', alpha=0.8)
axes[1, 1].set_yscale('log')
axes[1, 1].set_ylabel('Number of Customers (log)')
axes[1, 1].set_xlabel('Number of Purchases')
axes[1, 1].set_title('Frequency Distribution (Log scale)')
#MONETARY
# Monetary histogram
axes[2, 0].hist(rfm['Monetary'], bins=50, color=colors[3], edgecolor='black', alpha=0.8)
axes[2, 0].set_ylabel('Number of Customers')
axes[2, 0].set_xlabel('Total Spend (£)')
axes[2, 0].set_title('Monetary Distribution')
axes[2, 0].axvline(rfm['Monetary'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: £{rfm["Monetary"].median():.0f}')
axes[2, 0].legend()
# Monetary histogram (log scale)
axes[2, 1].hist(rfm['Monetary'], bins=50, color=colors[3], edgecolor='black', alpha=0.8)
axes[2, 1].set_yscale('log')
axes[2, 1].set_ylabel('Number of Customers (log)')
axes[2, 1].set_xlabel('Total Spend (£)')
axes[2, 1].set_title('Monetary Distribution (Log scale)')
plt.tight_layout()
plt.show()

# Display sample RFM data
print("\n" + "="*60)
print("Sample RFM Data (Top 5 Customers by Monetary (£) )")
print("="*60)
print(rfm.sort_values('Monetary', ascending=False).head(5).to_string(index=False))

4.Modeling

In [ ]:
#Modeling
#K-MEANS Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import time

rfm_features = rfm[['Recency', 'Frequency', 'Monetary']].copy()
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)
#optimal k
K_range = range(2, 10)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, kmeans.labels_))
    davies_bouldin_scores.append(davies_bouldin_score(rfm_scaled, kmeans.labels_))
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}, Davies-Bouldin={davies_bouldin_scores[-1]:.3f}")

#visualize metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Selecting Optimal K Metrics', fontsize=16, fontweight='bold')
# Elbow plot
axes[0].plot(K_range, inertias, marker='o', linewidth=2.5, markersize=9, color=colors[4])
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.35)
axes[0].set_xticks(K_range)
# Silhouette score (higher-better)
axes[1].plot(K_range, silhouette_scores, marker='o', linewidth=2.5, markersize=9, color=colors[0])
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_title('Silhouette Score')
axes[1].grid(True, alpha=0.35)
axes[1].set_xticks(K_range)
# DB score (lower-better)
axes[2].plot(K_range, davies_bouldin_scores, marker='o', linewidth=2.5, markersize=9, color=colors[1])
axes[2].set_ylabel('Davies-Bouldin Score')
axes[2].set_xlabel('Number of Clusters (K)')
axes[2].set_title('Davies-Bouldin Score')
axes[2].grid(True, alpha=0.35)
axes[2].set_xticks(K_range)

plt.tight_layout()
plt.show()

In [ ]:
#Modeling
#K-MEANS Clustering at optimal k=5
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
rfm['Kmeans_Cluster'] = kmeans_final.fit_predict(rfm_scaled)
print('K-Means Clustering (K=5) results:')
#print(f"Final Inertia: {kmeans_final.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(rfm_scaled, rfm['Kmeans_Cluster']):.3f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(rfm_scaled, rfm['Kmeans_Cluster']):.3f}" )

In [ ]:
#Modeling
#Hierarchical Clustering- Bottom up/agglomerative
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

print('Hierarchical Clustering results:')
model = AgglomerativeClustering(n_clusters=5,linkage='ward')
labels = model.fit_predict(rfm_scaled)
rfm[f"Hierarchical_Cluster"] = labels
sil = silhouette_score(rfm_scaled, labels)
dbi = davies_bouldin_score(rfm_scaled, labels)
print(f"Silhouette Score: {sil:.3f}")
print(f"Davies-Bouldin Index: {dbi:.3f}" )

In [ ]:
#Hyperparameter tuning
from sklearn.model_selection import ParameterGrid
print("K-MEANS HYPERPARAMETER TUNING")
#parameter grid
param_grid = {'init': ['k-means++', 'random'], 'max_iter': [100, 300, 500]}
# Grid search
results = []
for params in ParameterGrid(param_grid):
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10, **params)
    labels = kmeans.fit_predict(rfm_scaled)
    silhouette = silhouette_score(rfm_scaled, labels)
    db_score = davies_bouldin_score(rfm_scaled, labels)
    results.append({'init': params['init'],'max_iter': params['max_iter'], 'silhouette': silhouette, 'davies_bouldin': db_score})

results_df = pd.DataFrame(results).sort_values('silhouette', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
#Modeling
#Final Optimized K-MEANS Clustering
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10, init='k-means++',max_iter=100)
rfm['Optimized_Kmeans_Cluster'] = kmeans_final.fit_predict(rfm_scaled)
print('Final Optimized K-Means Clustering:')
#print(f"Final Inertia: {kmeans_final.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(rfm_scaled, rfm['Optimized_Kmeans_Cluster']):.3f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(rfm_scaled, rfm['Optimized_Kmeans_Cluster']):.3f}" )

In [ ]:
# Calculate cluster statistics
cluster_summary = rfm.groupby('Optimized_Kmeans_Cluster').agg({'Recency':['mean', 'median'],
    'Frequency': ['mean', 'median'],
    'Monetary': ['mean', 'median', 'sum'],
    'CustomerID': 'count'}).round(2)
cluster_summary.columns = ['_'.join(col).strip() for col in cluster_summary.columns.values]
cluster_summary.rename(columns={'CustomerID_count': 'CustomerCount'}, inplace=True)
cluster_summary['RevenuePercent'] = (cluster_summary['Monetary_sum'] / cluster_summary['Monetary_sum'].sum() * 100).round(2)
#print("Cluster Summary Statistics:")
#print(cluster_summary)
# Analyze each cluster
print("Interpreting clusters")
for cluster in range(optimal_k):
    cluster_data = rfm[rfm['Optimized_Kmeans_Cluster'] == cluster]
    print(f"\n--- CLUSTER {cluster} ---")
    print(f"Size: {len(cluster_data):,} customers ({len(cluster_data)/len(rfm)*100:.1f}%)")
    print(f"Recency: {cluster_data['Recency'].mean():.1f} days (avg)")
    print(f"Frequency: {cluster_data['Frequency'].mean():.1f} purchases (avg)")
    print(f"Monetary: £{cluster_data['Monetary'].mean():,.2f} (avg)")
    print(f"Total Revenue: £{cluster_data['Monetary'].sum():,.2f}")

In [ ]:
#Visualize customer segments
#3D scatter plot
fig3d = plt.figure(figsize=(12, 9))
ax1 = fig3d.add_subplot(111, projection='3d')
scatter = ax1.scatter(rfm['Recency'], rfm['Frequency'], rfm['Monetary'],c=rfm['Kmeans_Cluster'], cmap='viridis', s=50, alpha=0.6)
ax1.set_xlabel('Recency (days)')
ax1.set_ylabel('Frequency')
ax1.set_zlabel('Monetary (£)', labelpad=10)
ax1.set_title('3D Customer Segments')
plt.colorbar(scatter, ax=ax1, label='Cluster')
plt.show()

In [ ]:
#2D plots
fig2d, axes = plt.subplots(2, 2, figsize=(15, 12))
# Frequency vs Monetary
ax2 = axes[0, 0]
for cluster in range(optimal_k):
    cluster_data = rfm[rfm['Kmeans_Cluster'] == cluster]
    ax2.scatter(cluster_data['Frequency'], cluster_data['Monetary'],label=f'Cluster {cluster}', s=50, alpha=0.7)
ax2.set_ylabel('Monetary')
ax2.set_xlabel('Frequency')
ax2.set_title('Frequency vs Monetary')
ax2.legend()
ax2.grid(True, alpha=0.35)

#Recency vs Monetary
ax3 = axes[0, 1]
for cluster in range(optimal_k):
    cluster_data = rfm[rfm['Kmeans_Cluster'] == cluster]
    ax3.scatter(cluster_data['Recency'], cluster_data['Monetary'],label=f'Cluster {cluster}', s=50, alpha=0.7)
ax3.set_ylabel('Monetary')
ax3.set_xlabel('Recency (Days)')
ax3.set_title('Recency vs Monetary')
ax3.legend()
ax3.grid(True, alpha=0.35)

#Recency vs Frequency
ax4 = axes[1, 0]
for cluster in range(optimal_k):
    cluster_data = rfm[rfm['Kmeans_Cluster'] == cluster]
    ax4.scatter(cluster_data['Recency'], cluster_data['Frequency'],label=f'Cluster {cluster}', s=50, alpha=0.7)
ax4.set_ylabel('Frequency')
ax4.set_xlabel('Recency (Days)')
ax4.set_title('Recency vs Frequency')
ax4.legend()
ax4.grid(True, alpha=0.35)

axes[1, 1].axis('off')
plt.tight_layout()
plt.show()